# INF01090 - Ciência de Dados

# Lab Task 02 — Data Cleaning + Data Transformation (up to 4 people)

## Integrantes: Arthur Andrade da Silva, Cristopher de Wallau, Vítor Santana Feijó

## Goal

Your team must turn a messy dataset into a **clean, consistent, and analysis-ready dataset**.

## Deliverables

By the end of the lab, your group should produce:

1. a cleaned dataset
2. a transformed dataset
3. a short written justification of your choices
4. a competition submission for **best analysis-ready dataset**

## Important

This lab is about:

- identifying data quality problems
- cleaning the data
- transforming variables
- documenting your decisions



## Suggested workflow

### Part A — Inspect
Look at the dataset carefully before changing anything.

### Part B — Clean
Fix duplicates, categories, dates, missing values, and invalid values.

### Part C — Transform
Create a small set of useful transformed variables.

### Part D — Compete
Submit the dataset your team believes is the **best prepared for future analysis**.


In [61]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import altair as alt
np.random.seed(42)
pd.set_option("display.max_columns", None)


## 1. Dataset

This synthetic dataset contains several realistic problems:

- duplicate rows
- inconsistent text categories
- mixed date formats
- missing values
- invalid ages
- numeric values stored as text
- one suspicious high-income value

Your task is to prepare the dataset for future analysis.


In [62]:
df_raw = pd.DataFrame({
    "customer_id": [101, 102, 103, 103, 104, 105, 106, 107, 108, 109, 110, 111],
    "name": ["Ana", " Bruno", "carla", "carla", "Daniel ", "Eva", None, "GUSTAVO", "Helena", "Igor", "Julia", "Julia"],
    "age": [25, 31, 200, 200, 29, np.nan, 41, -3, 36, 28, 28, 28],
    "city": ["Porto Alegre", "porto alegre", "São Paulo", "Sao Paulo", "Rio de Janeiro", "RIO DE JANEIRO", "Curitiba", "Curitiba ", None, "Porto Alegre", "São Paulo", "São Paulo"],
    "income": ["4200", "5100", "7900", "7900", "not informed", "6100", "5800", "4700", "250000", None, "4300", "4300"],
    "signup_date": ["2024-01-10", "2024/02/15", "15-03-2024", "15-03-2024", "2024-04-01", "2024-13-05", None, "2024-06-20", "2024-07-01", "2024-07-32", "2024-08-10", "2024-08-10"],
    "purchases": [3, 5, 7, 7, 2, 4, np.nan, 1, 30, 2, 3, 3],
    "gender": ["F", "M", "F", "F", "M", "F", "M", "m", "Female", "M", "F", "F"],
    "segment": ["Basic", "Basic", "Premium", "Premium", "Basic", "Premium", "Basic", "Basic", "Premium", "Basic", "Basic", "Basic"]
})

df_raw


,customer_id,name,age,city,income,signup_date,purchases,gender,segment
0,101,Ana,25.0,Porto Alegre,4200,2024-01-10,3.0,F,Basic
1,102,Bruno,31.0,porto alegre,5100,2024/02/15,5.0,M,Basic
2,103,carla,200.0,São Paulo,7900,15-03-2024,7.0,F,Premium
3,103,carla,200.0,Sao Paulo,7900,15-03-2024,7.0,F,Premium
4,104,Daniel,29.0,Rio de Janeiro,not informed,2024-04-01,2.0,M,Basic
5,105,Eva,NaN,RIO DE JANEIRO,6100,2024-13-05,4.0,F,Premium
6,106,None,41.0,Curitiba,5800,None,NaN,M,Basic
7,107,GUSTAVO,-3.0,Curitiba,4700,2024-06-20,1.0,m,Basic
8,108,Helena,36.0,None,250000,2024-07-01,30.0,Female,Premium
9,109,Igor,28.0,Porto Alegre,None,2024-07-32,2.0,M,Basic


## 2. Initial inspection

Inspect the dataset before changing it.


In [63]:
df_raw.shape


(12, 9)

In [64]:
df_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  12 non-null     int64  
 1   name         11 non-null     object 
 2   age          11 non-null     float64
 3   city         11 non-null     object 
 4   income       11 non-null     object 
 5   signup_date  11 non-null     object 
 6   purchases    11 non-null     float64
 7   gender       12 non-null     object 
 8   segment      12 non-null     object 
dtypes: float64(2), int64(1), object(6)
memory usage: 996.0+ bytes


In [65]:
df_raw.isna().sum()


,0
customer_id,0
name,1
age,1
city,1
income,1
signup_date,1
purchases,1
gender,0
segment,0


In [66]:
df_raw.describe(include="all")


,customer_id,name,age,city,income,signup_date,purchases,gender,segment
count,12.000000,11,11.000000,11,11,11,11.000000,12,12
unique,NaN,9,NaN,8,9,9,NaN,4,2
top,NaN,carla,NaN,São Paulo,7900,15-03-2024,NaN,F,Basic
freq,NaN,2,NaN,3,2,2,NaN,6,8
mean,105.750000,NaN,58.454545,NaN,NaN,NaN,6.090909,NaN,NaN
std,3.278719,NaN,70.836945,NaN,NaN,NaN,8.166450,NaN,NaN
min,101.000000,NaN,-3.000000,NaN,NaN,NaN,1.000000,NaN,NaN
25%,103.000000,NaN,28.000000,NaN,NaN,NaN,2.500000,NaN,NaN
50%,105.500000,NaN,29.000000,NaN,NaN,NaN,3.000000,NaN,NaN
75%,108.250000,NaN,38.500000,NaN,NaN,NaN,6.000000,NaN,NaN


## 3. Create a working copy


In [67]:
df = df_raw.copy()


# Part A — Core Cleaning Tasks


## 4. Standardize text columns

Clean the following:

- `name`
- `city`
- `gender`

Suggestions:
- strip extra spaces
- normalize capitalization
- standardize labels


In [68]:
import pandas as pd
import numpy as np
# 1. Definir o mapeamento de correção para as cidades
mapeamento_cidades = {
    'Sao Paulo': 'São Paulo',
    'Sao paulo': 'São Paulo',
    'Sāo Paulo': 'São Paulo',
    'Porto Alegre': 'Porto Alegre',
    'Curitiba': 'Curitiba',
    'Rio De Janeiro': 'Rio de Janeiro' # Ajustando o 'de' para minúsculo
}

# 2. Primeiro, limpamos os espaços e padronizamos o case
df['city'] = df['city'].str.strip().str.title()

# 3. Aplicamos o mapeamento para colocar os acentos corretos
# O .replace() aqui é melhor que o .map() porque ele mantém o valor original
# caso a cidade não esteja no dicionário.
df['city'] = df['city'].replace(mapeamento_cidades)
# 1. Substitui strings de preenchimento por NaN real (sem excluir linhas)
df.replace(['None', 'not informed'], np.nan, inplace=True)
# 2. Normaliza os nomes ,generos e cidades para o padrão 'Nome Sobrenome'
df['name'] = df['name'].str.strip().str.title()
# 3. Salva o arquivo para continuar depois
df.to_csv('clientes_processados.csv', index=False)

# Exibe o dataframe resultante
print(df)

    customer_id     name    age            city  income signup_date  \
0           101      Ana   25.0    Porto Alegre    4200  2024-01-10   
1           102    Bruno   31.0    Porto Alegre    5100  2024/02/15   
2           103    Carla  200.0       São Paulo    7900  15-03-2024   
3           103    Carla  200.0       São Paulo    7900  15-03-2024   
4           104   Daniel   29.0  Rio de Janeiro     NaN  2024-04-01   
5           105      Eva    NaN  Rio de Janeiro    6100  2024-13-05   
6           106     None   41.0        Curitiba    5800        None   
7           107  Gustavo   -3.0        Curitiba    4700  2024-06-20   
8           108   Helena   36.0            None  250000  2024-07-01   
9           109     Igor   28.0    Porto Alegre    None  2024-07-32   
10          110    Julia   28.0       São Paulo    4300  2024-08-10   
11          111    Julia   28.0       São Paulo    4300  2024-08-10   

    purchases  gender  segment  
0         3.0       F    Basic  
1         

In [69]:
# TODO: standardize gender values
mapeamento_genero = {
    'F': 'F',
    'Female': 'F',
    'f': 'F',
    'female': 'F',
    'M': 'M',
    'm': 'M',
    'Male': 'M',
    'male': 'M'
}
df['gender'] = df['gender'].str.strip().map(mapeamento_genero)
print(df)

    customer_id     name    age            city  income signup_date  \
0           101      Ana   25.0    Porto Alegre    4200  2024-01-10   
1           102    Bruno   31.0    Porto Alegre    5100  2024/02/15   
2           103    Carla  200.0       São Paulo    7900  15-03-2024   
3           103    Carla  200.0       São Paulo    7900  15-03-2024   
4           104   Daniel   29.0  Rio de Janeiro     NaN  2024-04-01   
5           105      Eva    NaN  Rio de Janeiro    6100  2024-13-05   
6           106     None   41.0        Curitiba    5800        None   
7           107  Gustavo   -3.0        Curitiba    4700  2024-06-20   
8           108   Helena   36.0            None  250000  2024-07-01   
9           109     Igor   28.0    Porto Alegre    None  2024-07-32   
10          110    Julia   28.0       São Paulo    4300  2024-08-10   
11          111    Julia   28.0       São Paulo    4300  2024-08-10   

    purchases gender  segment  
0         3.0      F    Basic  
1         5.

In [70]:
# TODO: inspect unique values after standardization
print(df.value_counts(dropna=False))

customer_id  name     age     city            income  signup_date  purchases  gender  segment
103          Carla     200.0  São Paulo       7900    15-03-2024   7.0        F       Premium    2
101          Ana       25.0   Porto Alegre    4200    2024-01-10   3.0        F       Basic      1
102          Bruno     31.0   Porto Alegre    5100    2024/02/15   5.0        M       Basic      1
104          Daniel    29.0   Rio de Janeiro  NaN     2024-04-01   2.0        M       Basic      1
105          Eva       NaN    Rio de Janeiro  6100    2024-13-05   4.0        F       Premium    1
106          NaN       41.0   Curitiba        5800    NaN          NaN        M       Basic      1
107          Gustavo  -3.0    Curitiba        4700    2024-06-20   1.0        M       Basic      1
108          Helena    36.0   NaN             250000  2024-07-01   30.0       F       Premium    1
109          Igor      28.0   Porto Alegre    NaN     2024-07-32   2.0        M       Basic      1
110          Ju

## 5. Parse dates with mixed formats

The dataset contains valid dates written in multiple formats:

- `YYYY-MM-DD`
- `YYYY/MM/DD`
- `DD-MM-YYYY`

You should preserve valid information and normalize it into a unified format.


In [71]:
# TODO: complete this helper and apply it to signup_date

def parse_mixed_date(x):
    if pd.isna(x):
        return pd.NaT

    x = str(x).strip()
    if x == "":
        return pd.NaT

    formats = [
        "%Y-%m-%d",
        "%Y/%m/%d",
        "%d-%m-%Y",
    ]

    for fmt in formats:
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue

    return pd.to_datetime(x, errors="coerce")
df['signup_date'] = df['signup_date'].apply(parse_mixed_date)

print(df)

    customer_id     name    age            city  income signup_date  \
0           101      Ana   25.0    Porto Alegre    4200  2024-01-10   
1           102    Bruno   31.0    Porto Alegre    5100  2024-02-15   
2           103    Carla  200.0       São Paulo    7900  2024-03-15   
3           103    Carla  200.0       São Paulo    7900  2024-03-15   
4           104   Daniel   29.0  Rio de Janeiro     NaN  2024-04-01   
5           105      Eva    NaN  Rio de Janeiro    6100         NaT   
6           106     None   41.0        Curitiba    5800         NaT   
7           107  Gustavo   -3.0        Curitiba    4700  2024-06-20   
8           108   Helena   36.0            None  250000  2024-07-01   
9           109     Igor   28.0    Porto Alegre    None         NaT   
10          110    Julia   28.0       São Paulo    4300  2024-08-10   
11          111    Julia   28.0       São Paulo    4300  2024-08-10   

    purchases gender  segment  
0         3.0      F    Basic  
1         5.

## 6. Remove exact duplicates

Inspect duplicate rows and then remove them.


In [72]:
# TODO: count duplicated rows
total_duplicates = df.duplicated().sum()
print(total_duplicates)

1


In [73]:
# TODO: display duplicated rows
# Filtra o DataFrame mostrando apenas as linhas que se repetem
print(df[df.duplicated(keep=False)])
# O 'keep=False' mostra todas as cópias.
# Se usar sem ele, mostra apenas da segunda cópia em diante.

   customer_id   name    age       city income signup_date  purchases gender  \
2          103  Carla  200.0  São Paulo   7900  2024-03-15        7.0      F   
3          103  Carla  200.0  São Paulo   7900  2024-03-15        7.0      F   

   segment  
2  Premium  
3  Premium  


In [74]:
# TODO: remove exact duplicates
df = df.drop_duplicates()

In [75]:
print(df)

    customer_id     name    age            city  income signup_date  \
0           101      Ana   25.0    Porto Alegre    4200  2024-01-10   
1           102    Bruno   31.0    Porto Alegre    5100  2024-02-15   
2           103    Carla  200.0       São Paulo    7900  2024-03-15   
4           104   Daniel   29.0  Rio de Janeiro     NaN  2024-04-01   
5           105      Eva    NaN  Rio de Janeiro    6100         NaT   
6           106     None   41.0        Curitiba    5800         NaT   
7           107  Gustavo   -3.0        Curitiba    4700  2024-06-20   
8           108   Helena   36.0            None  250000  2024-07-01   
9           109     Igor   28.0    Porto Alegre    None         NaT   
10          110    Julia   28.0       São Paulo    4300  2024-08-10   
11          111    Julia   28.0       São Paulo    4300  2024-08-10   

    purchases gender  segment  
0         3.0      F    Basic  
1         5.0      M    Basic  
2         7.0      F  Premium  
4         2.0      

## 7. Convert numeric text and fix invalid ages

Tasks:
- convert `income` to numeric
- identify impossible ages
- replace impossible ages with missing values


In [76]:
# TODO: convert income to numeric
df['income'] = pd.to_numeric(df['income'], errors='coerce')

In [77]:
# TODO: identify invalid ages
# 1. Garante que a coluna é numérica
df['age'] = pd.to_numeric(df['age'], errors='coerce')

# 2. Define o que é uma idade válida (ex: entre 0 e 110 anos)
condition = (df['age'] >= 0) & (df['age'] <= 110)

In [78]:
# TODO: replace invalid ages with NaN
# Onde a condição NÃO for atendida, o .where coloca NaN
df['age'] = df['age'].where(condition, np.nan)
print(df)

    customer_id     name   age            city    income signup_date  \
0           101      Ana  25.0    Porto Alegre    4200.0  2024-01-10   
1           102    Bruno  31.0    Porto Alegre    5100.0  2024-02-15   
2           103    Carla   NaN       São Paulo    7900.0  2024-03-15   
4           104   Daniel  29.0  Rio de Janeiro       NaN  2024-04-01   
5           105      Eva   NaN  Rio de Janeiro    6100.0         NaT   
6           106     None  41.0        Curitiba    5800.0         NaT   
7           107  Gustavo   NaN        Curitiba    4700.0  2024-06-20   
8           108   Helena  36.0            None  250000.0  2024-07-01   
9           109     Igor  28.0    Porto Alegre       NaN         NaT   
10          110    Julia  28.0       São Paulo    4300.0  2024-08-10   
11          111    Julia  28.0       São Paulo    4300.0  2024-08-10   

    purchases gender  segment  
0         3.0      F    Basic  
1         5.0      M    Basic  
2         7.0      F  Premium  
4      

## 8. Handle missing values

For this lab, use a simple strategy unless your group has a justified alternative.

Suggestion:
- median for numerical variables
- `"Unknown"` for selected categorical variables


In [79]:
# TODO: inspect missing values

print(df.isnull().sum())

customer_id    0
name           1
age            3
city           1
income         2
signup_date    3
purchases      1
gender         0
segment        0
dtype: int64


In [80]:
# TODO: fill missing values

#NUMERIC VARIABLES
# Calcula a mediana
median_age = df['age'].median()
median_income = df['income'].median()
median_purchases = df['purchases'].median()
median_data = df['signup_date'].median()
# Preenche os nulos diretamente
df['age'] = df['age'].fillna(median_age)
df['income'] = df['income'].fillna(median_income)
df['purchases'] = df['purchases'].fillna(median_purchases)
#CATEGORICAL VARIABLES
df['gender'] = df['gender'].fillna('Unknown')
df['name'] = df['name'].fillna('Unknown')
df['city'] = df['city'].fillna('Unknown')
df['segment'] = df['segment'].fillna('Unknown')
#DATA VARIABLES
df['signup_date'] = df['signup_date'].fillna(median_data)
print(df)

    customer_id     name   age            city    income signup_date  \
0           101      Ana  25.0    Porto Alegre    4200.0  2024-01-10   
1           102    Bruno  31.0    Porto Alegre    5100.0  2024-02-15   
2           103    Carla  28.5       São Paulo    7900.0  2024-03-15   
4           104   Daniel  29.0  Rio de Janeiro    5100.0  2024-04-01   
5           105      Eva  28.5  Rio de Janeiro    6100.0  2024-05-11   
6           106  Unknown  41.0        Curitiba    5800.0  2024-05-11   
7           107  Gustavo  28.5        Curitiba    4700.0  2024-06-20   
8           108   Helena  36.0         Unknown  250000.0  2024-07-01   
9           109     Igor  28.0    Porto Alegre    5100.0  2024-05-11   
10          110    Julia  28.0       São Paulo    4300.0  2024-08-10   
11          111    Julia  28.0       São Paulo    4300.0  2024-08-10   

    purchases gender  segment  
0         3.0      F    Basic  
1         5.0      M    Basic  
2         7.0      F  Premium  
4      

## 9. Inspect the suspicious income outlier

There is one very large income value.

Tasks:
- make a boxplot
- inspect the suspicious row
- decide whether to keep, cap, or otherwise treat it
- justify your choice later


In [81]:
# TODO: create a boxplot for income_num
# retiramos o outlier da visualização do boxplot para melhor interpretação.
# Decidimos não remove-lo do dataframe pois pode ser um cliente válido, apenas com uma renda alta.
df_plot = df[df['income'] <= 50000]

altair_chart = alt.Chart(df_plot).mark_boxplot().encode(
    y='income'
).properties(
    title='Boxplot of Income (<= 50000)'
)
altair_chart

alt.Chart(...)

# Part B — Core Transformation Tasks


## 10. Create transformed variables

Create the following:

- `signup_month`
- `income_minmax`
- `income_log`
- `segment_encoded`
- `purchases_per_age`
- `income_per_purchase`


In [82]:
# TODO: create the transformed columns listed above
df['signup_month'] = df['signup_date'].dt.month
df['income_minmax'] = (df['income'] - df['income'].min()) / (df['income'].max() - df['income'].min())
df['income_log'] = np.log(df['income'] + 1)
df['segment_encoded'] = df['segment'].map({'Basic': 0, 'Premium': 1})
df['purchases_per_age'] = df['purchases'] / df['age']
df['income_per_purchase'] = df['income'] / df['purchases']

print(df.columns)

Index(['customer_id', 'name', 'age', 'city', 'income', 'signup_date',
       'purchases', 'gender', 'segment', 'signup_month', 'income_minmax',
       'income_log', 'segment_encoded', 'purchases_per_age',
       'income_per_purchase'],
      dtype='object')


## 11. One-hot encode one categorical variable

Use one-hot encoding for the `city` column.


In [83]:
# TODO: create city dummy variables
one_hot_cities = pd.get_dummies(df['city'], prefix='city')
df = pd.concat([df, one_hot_cities], axis=1)
print(df)

    customer_id     name   age            city    income signup_date  \
0           101      Ana  25.0    Porto Alegre    4200.0  2024-01-10   
1           102    Bruno  31.0    Porto Alegre    5100.0  2024-02-15   
2           103    Carla  28.5       São Paulo    7900.0  2024-03-15   
4           104   Daniel  29.0  Rio de Janeiro    5100.0  2024-04-01   
5           105      Eva  28.5  Rio de Janeiro    6100.0  2024-05-11   
6           106  Unknown  41.0        Curitiba    5800.0  2024-05-11   
7           107  Gustavo  28.5        Curitiba    4700.0  2024-06-20   
8           108   Helena  36.0         Unknown  250000.0  2024-07-01   
9           109     Igor  28.0    Porto Alegre    5100.0  2024-05-11   
10          110    Julia  28.0       São Paulo    4300.0  2024-08-10   
11          111    Julia  28.0       São Paulo    4300.0  2024-08-10   

    purchases gender  segment  signup_month  income_minmax  income_log  \
0         3.0      F    Basic             1       0.000000   

## 12. Build the final transformed dataset

Create a compact final table ready for future analysis.
Include:
- cleaned core columns
- transformed columns
- one-hot encoded city columns


In [84]:
# TODO: build df_final

colunas_originais_limpas = ['customer_id', 'name', 'age', 'income', 'signup_date', 'purchases', 'gender', 'segment']
colunas_transformadas = ['signup_month', 'income_minmax', 'income_log', 'segment_encoded', 'purchases_per_age', 'income_per_purchase']

colunas_city_dummies = [c for c in df.columns if c.startswith('city_')]

todas_colunas_finais = colunas_originais_limpas + colunas_transformadas + colunas_city_dummies

df_final = df[todas_colunas_finais]

print("Dataset final pronto para análise:")
display(df_final.head(11))

Dataset final pronto para análise:


,customer_id,name,age,income,signup_date,purchases,gender,segment,signup_month,income_minmax,income_log,segment_encoded,purchases_per_age,income_per_purchase,city_Curitiba,city_Porto Alegre,city_Rio de Janeiro,city_São Paulo,city_Unknown
0,101,Ana,25.0,4200.0,2024-01-10,3.0,F,Basic,1,0.000000,8.343078,0,0.120000,1400.000000,False,True,False,False,False
1,102,Bruno,31.0,5100.0,2024-02-15,5.0,M,Basic,2,0.003662,8.537192,0,0.161290,1020.000000,False,True,False,False,False
2,103,Carla,28.5,7900.0,2024-03-15,7.0,F,Premium,3,0.015053,8.974745,1,0.245614,1128.571429,False,False,False,True,False
4,104,Daniel,29.0,5100.0,2024-04-01,2.0,M,Basic,4,0.003662,8.537192,0,0.068966,2550.000000,False,False,True,False,False
5,105,Eva,28.5,6100.0,2024-05-11,4.0,F,Premium,5,0.007730,8.716208,1,0.140351,1525.000000,False,False,True,False,False
6,106,Unknown,41.0,5800.0,2024-05-11,3.0,M,Basic,5,0.006509,8.665786,0,0.073171,1933.333333,True,False,False,False,False
7,107,Gustavo,28.5,4700.0,2024-06-20,1.0,M,Basic,6,0.002034,8.455531,0,0.035088,4700.000000,True,False,False,False,False
8,108,Helena,36.0,250000.0,2024-07-01,30.0,F,Premium,7,1.000000,12.429220,1,0.833333,8333.333333,False,False,False,False,True
9,109,Igor,28.0,5100.0,2024-05-11,2.0,M,Basic,5,0.003662,8.537192,0,0.071429,2550.000000,False,True,False,False,False
10,110,Julia,28.0,4300.0,2024-08-10,3.0,F,Basic,8,0.000407,8.366603,0,0.107143,1433.333333,False,False,False,True,False


## 13. Optional summary table

Create a small grouped summary by city with:
- average income
- total purchases


In [85]:
# TODO: grouped summary by city


# Part C — Competition


## 14. Competition: Best Analysis-Ready Dataset

Your group will submit:

1. one final cleaned + transformed dataset
2. one short markdown explanation
3. one short “quality report”

### Judging criteria
- correctness
- consistency
- clarity
- usefulness for future analysis
- quality of justification

### Tip
A dataset is not “better” just because it has more columns.
It is better if it is **clean, meaningful, and well documented**.


## 15. Team explanation

In the markdown cell below, explain:

- which duplicate handling you used
- how you standardized categories
- how you parsed dates
- how you handled missing values
- what you did with the suspicious income value
- why your final transformed dataset is a good submission


**Write your team explanation here.**

##Relatório de Preparação de Dados

###Integrantes: Arthur Andrade da Silva, Cristopher de Wallau, Vítor Santana Feijó

1. Tratamento de Duplicatas

Identificamos e removemos as linhas que eram duplicatas exatas, como os registros repetidos da cliente "Carla" e da cliente "Julia". Mantivemos apenas a primeira ocorrência de cada registro para evitar que a análise estatística posterior fosse inflada artificialmente por dados redundantes.

2. Padronização de Categorias

Aplicamos um processo rigoroso de limpeza em três colunas principais:

Nome e Cidade: Removemos espaços em branco extras e padronizamos para Title Case (ex: " Bruno" → "Bruno"). Corrigimos inconsistências de acentuação e grafia manualmente via mapeamento (ex: "Sao Paulo" → "São Paulo" e "RIO DE JANEIRO" → "Rio de Janeiro").

Gênero: Consolidamos as diversas entradas inconsistentes (m, M, Female, F) em um padrão simplificado de M e F.

Segmento: Além de manter a categoria original, criamos a coluna segment_encoded (0 para Basic, 1 para Premium) para permitir o uso imediato em modelos matemáticos.

3. Parsing de Datas com Formatos Mistos

O dataset original possuía três formatos distintos de data (YYYY-MM-DD, YYYY/MM/DD, DD-MM-YYYY). Desenvolvemos uma função auxiliar que testou cada formato, convertendo todos para o padrão unificado datetime do pandas. Datas inválidas (como o dia "32") foram transformadas em NaT (Not a Time) e posteriormente tratadas para não interromper o fluxo de análise.

4. Tratamento de Valores Ausentes e Inválidos

Adotamos estratégias diferenciadas conforme a natureza do dado para garantir a integridade do dataset:

Numéricos (age, income, purchases): Utilizamos a mediana para o preenchimento de nulos, pois ela é mais resistente a valores extremos (outliers) do que a média.

Categóricos (name, city): Preenchemos valores ausentes com o rótulo "Unknown", preservando as linhas para análise de outras variáveis.

Idades Inválidas: Valores biologicamente impossíveis (como -3.0 ou 200.0) foram detectados e substituídos pela mediana da idade do grupo.

5. Análise do Outlier de Renda

Detectamos um valor de renda de 250.000, que se distanciava drasticamente da distribuição dos demais clientes (média em torno de 5.000).

Decisão: Optamos por manter o registro no dataset.

Justificativa: Em contextos reais de ciência de dados, clientes de altíssima renda são pontos de dados válidos e valiosos para segmentação. Para mitigar o impacto desse valor em modelos sensíveis, criamos a variável income_log, que "achata" a escala e normaliza a distribuição.

6. Por que este dataset é a melhor submissão?

Nosso df_final foi construído sem a retirada de nenhuma coluna para ser versátil, robusto e completo:

1. Versatilidade para Diferentes Algoritmos

Nem todo modelo matemático processa dados da mesma forma. Ao oferecer múltiplas versões, o dataset torna-se "universal".

2. Equilíbrio entre Interpretabilidade e Performance

Interpretabilidade Humana: A coluna original income é essencial para validação, relatórios de negócio e apresentações, pois valores como "50.000" são compreensíveis, enquanto um logaritmo de "10.8" não é.

Performance da Máquina: O income_log suaviza o impacto de outliers (como o valor de 250.000), permitindo que o modelo aprenda padrões gerais sem ser distorcido por um único cliente muito rico.

3. Segurança e Rastreabilidade

Manter a coluna original limpa permite realizar o "caminho de volta" e auditar os dados. Se apenas a versão transformada for mantida e houver um erro no cálculo, perde-se a base de comparação e a integridade da informação original.


## 16. Quick checklist before submission

Make sure your group has:

- removed exact duplicates
- standardized text categories
- parsed dates into one format
- converted income to numeric
- handled invalid ages
- handled missing values
- created the required transformed columns
- created one-hot encoded city columns
- built a final dataset
- written a short justification


# Final takeaway

A dataset becomes useful through a sequence of careful decisions.

In this lab, the main goal is not to “finish fast,” but to produce a dataset that is:

- correct
- consistent
- clearly documented
- ready for future analysis
